In [1]:
import numpy as np
from numba import jit,njit, prange
import time
import forallpeople as si
from steel_lib.experimental_functions import optional_reporting_handcalc
si.environment("default")

# A computationally intensive function in pure Python
def sum_squared_diff_python(x, y):
    """
    Calculates the sum of squared differences between two arrays.
    """
    result = 0
    for i in range(len(x)):
        result += (x[i] - y[i])**2
    return result

# The same function accelerated with Numba's @jit decorator
@optional_reporting_handcalc(config_object=None,key=None, detailed=False)
def sum_squared_diff_numba(x, y):
    """
    Calculates the sum of squared differences between two arrays using Numba.
    The @jit(nopython=True) decorator compiles this function to machine code.
    """
    result = 0 
    for i in prange(len(x)):
        result += (x[i]  - y[i]  )**2
    return result

# The same function accelerated with Numba's @jit decorator
@jit(nopython=True)
def sum_squared_diff_numba_1(x, y):
    """
    Calculates the sum of squared differences between two arrays using Numba.
    The @jit(nopython=True) decorator compiles this function to machine code.
    """
    result = 0 
    for i in prange(len(x)):
        result += (x[i]  - y[i]  )**2
    return result
# Create two large NumPy arrays with random data
array1 = np.random.rand(10_000_000)
array2 = np.random.rand(10_000_000)

In [2]:
# %%timeit
# result_python = sum_squared_diff_python(array1, array2)

In [3]:
# %%timeit
# result_numba = sum_squared_diff_numba(array1, array2)

In [4]:
# %%timeit
# result_numba_1 = sum_squared_diff_numba_1(array1, array2)

In [5]:
# import optuna

# def objective(trial):
#     # 1. Dynamically suggest the number of different members
#     num_beams = trial.suggest_int('num_beams', 1, 3)
#     num_plates = trial.suggest_int('num_plates', 2, 6)

#     # Dictionary to hold all the dimensions
#     design_parameters = {
#         'beams': [],
#         'plates': []
#     }

#     # 2. Dynamically define parameters for each beam
#     for i in range(num_beams):
#         beam_depth = trial.suggest_float(f'beam_{i}_depth', 200, 500)
#         beam_width = trial.suggest_float(f'beam_{i}_width', 100, 300)
#         design_parameters['beams'].append({'depth': beam_depth, 'width': beam_width})

#     # 3. Dynamically define parameters for each plate
#     for i in range(num_plates):
#         plate_thickness = trial.suggest_float(f'plate_{i}_thickness', 5, 20)
#         plate_material = trial.suggest_categorical(f'plate_{i}_material', ['Steel', 'Aluminum'])
#         design_parameters['plates'].append({'thickness': plate_thickness, 'material': plate_material})

#     # 4. Conditional parameter for a support type
#     support_type = trial.suggest_categorical('support_type', ['fixed', 'pinned'])
#     if support_type == 'fixed':
#         moment_capacity = trial.suggest_float('support_moment_capacity', 100, 1000)
#         design_parameters['support'] = {'type': 'fixed', 'moment_capacity': moment_capacity}
#     else:
#         design_parameters['support'] = {'type': 'pinned'}


#     # 5. Your structural analysis/simulation function would go here
#     # This function takes the design_parameters and returns a performance metric
#     # For example, it could be the structural efficiency, cost, or failure load.
#     # performance_metric = run_structural_analysis(design_parameters)

#     # For this example, let's assume a dummy performance metric
#     performance_metric = -1 * (sum(b['depth'] for b in design_parameters['beams']) +
#                               sum(p['thickness'] for p in design_parameters['plates']))


#     return performance_metric

# # Create a study object and optimize the objective function
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=10000)

# print("Best trial:")
# trial = study.best_trial

# print(f"  Value: {trial.value}")
# print("  Params: ")
# for key, value in trial.params.items():
#     print(f"    {key}: {value}")

In [6]:
import numpy as np
import time
from steel_lib.experimental_functions import bolt_shear,bolt_bearing
from numba import prange, jit

# Set the number of test cases
num_cases = 10000000

# Generate random primitive data for the test
F_nv_values = np.random.uniform(400, 600, size=num_cases).astype(np.float64)
A_bolt_values = np.random.uniform(0.5, 2.0, size=num_cases).astype(np.float64)
N_shear_planes_values = np.random.randint(1, 3, size=num_cases).astype(np.int64)
phi_values = np.full(num_cases, 0.75, dtype=np.float64)

# Array to store the results
results = np.zeros(num_cases, dtype=np.float64)

@jit(nopython=True, parallel=True,cache=True)
def test_function(F_nv_values,A_bolt_values,N_shear_planes_values,phi_values,results,num_cases):
    for i in prange(num_cases):
        results[i] = bolt_shear(
            F_nv=F_nv_values[i],
            A_bolt=A_bolt_values[i],
            N_shear_planes=N_shear_planes_values[i],
            phi=phi_values[i]
        )
    return results

In [7]:
%%timeit
test_function(F_nv_values,A_bolt_values,N_shear_planes_values,phi_values,results,num_cases) 

19.8 ms ± 768 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
import numpy as np
#create dummy array
from handcalcs.decorator import handcalc
a = np.random.rand(10)


In [9]:
@handcalc(jupyter_display=True)
def sumasd():
    aasdasd = a[1:3]
    aasd =  sum(a[1:3])
sumasd()

<IPython.core.display.Latex object>

In [10]:
import numpy as np
from numba import njit

# Define a custom structured dtype
particle_dtype = np.dtype([('id', np.int32), ('velocity', np.float64), ('position', np.float64)])

@njit
def process_particles(particles):
    for p in particles:
        p.position += p.velocity # Access fields by name

# Create an array with this custom type
data = np.zeros(10, dtype=particle_dtype)
asdasd = process_particles(data)

In [1]:

import numpy as np
import time
from steel_lib.experimental_functions import bolt_bearing, bolt_shear
from numba import prange, jit

# Set the number of test cases
num_cases = 1000000

# --- Generate random data for bolt_bearing ---
F_u_values = np.random.uniform(400, 600, size=num_cases).astype(np.float64)
d_bolt_values = np.random.uniform(16, 24, size=num_cases).astype(np.float64)
t_values = np.random.uniform(5, 20, size=num_cases).astype(np.float64)
P_u_values = np.random.uniform(100, 500, size=num_cases).astype(np.float64)
V_u_values = np.random.uniform(50, 300, size=num_cases).astype(np.float64)
S_r_values = np.random.uniform(50, 100, size=num_cases).astype(np.float64)
N_r_values = np.random.randint(2, 6, size=num_cases).astype(np.int64)
S_c_values = np.random.uniform(50, 100, size=num_cases).astype(np.float64)
N_c_values = np.random.randint(2, 4, size=num_cases).astype(np.int64)
L_ev_values = np.random.uniform(30, 60, size=num_cases).astype(np.float64)
L_eh_values = np.random.uniform(30, 60, size=num_cases).astype(np.float64)
dv_values = np.random.uniform(18, 26, size=num_cases).astype(np.float64)
dh_values = np.random.uniform(18, 26, size=num_cases).astype(np.float64)
phi_values = np.full(num_cases, 0.75, dtype=np.float64)

# --- Generate random data for bolt_shear ---
F_nv_values = np.random.uniform(400, 600, size=num_cases).astype(np.float64)
A_bolt_values = np.random.uniform(0.5, 2.0, size=num_cases).astype(np.float64)
N_shear_planes_values = np.random.randint(1, 3, size=num_cases).astype(np.int64)


# Array to store the results
results_bearing = np.zeros(num_cases, dtype=np.float64)
results_shear = np.zeros(num_cases, dtype=np.float64)

@jit(nopython=True, parallel=True, cache=True)
def test_bolt_strength_function(
    # Bearing args
    F_u_values, d_bolt_values, t_values, P_u_values, V_u_values, S_r_values, N_r_values, S_c_values, N_c_values, L_ev_values, L_eh_values, dv_values, dh_values,
    # Shear args
    F_nv_values, A_bolt_values, N_shear_planes_values,
    # Common args
    phi_values, 
    # Result arrays
    results_bearing, results_shear,
    # Other
    num_cases):
    for i in prange(num_cases):
        # For simplicity in this test, we'll set N_c to be 1 for some cases to test that branch
        # In a real scenario, you might want to test different logic branches more systematically.
        n_c = 1 if i % 2 == 0 else N_c_values[i]
        results_bearing[i] = bolt_bearing(
            F_u=F_u_values[i],
            d_bolt=d_bolt_values[i],
            t=t_values[i],
            P_u=P_u_values[i],
            V_u=V_u_values[i],
            S_r=S_r_values[i],
            N_r=N_r_values[i],
            S_c=S_c_values[i],
            N_c=n_c,
            L_ev=L_ev_values[i],
            L_eh=L_eh_values[i],
            dv=dv_values[i],
            dh=dh_values[i],
            phi=phi_values[i]
        )
        results_shear[i] = bolt_shear(
            F_nv=F_nv_values[i],
            A_bolt=A_bolt_values[i],
            N_shear_planes=N_shear_planes_values[i],
            phi=phi_values[i]
        )
    return results_bearing, results_shear

# Time the execution
# start_time = time.time()
# test_bolt_strength_function(
#     F_u_values, d_bolt_values, t_values, P_u_values, V_u_values, S_r_values, N_r_values, S_c_values, N_c_values, L_ev_values, L_eh_values, dv_values, dh_values,
#     F_nv_values, A_bolt_values, N_shear_planes_values,
#     phi_values,
#     results_bearing, results_shear,
#     num_cases
# )
# end_time = time.time()

# print(f"Bolt bearing and shear test with {num_cases} cases took {end_time - start_time:.4f} seconds.")


In [3]:
%%timeit
test_bolt_strength_function(
    F_u_values, d_bolt_values, t_values, P_u_values, V_u_values, S_r_values, N_r_values, S_c_values, N_c_values, L_ev_values, L_eh_values, dv_values, dh_values,
    F_nv_values, A_bolt_values, N_shear_planes_values,
    phi_values,
    results_bearing, results_shear,
    num_cases
)

12.5 ms ± 628 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [1]:
from math import sin,cos,pi
from handcalcs.decorator import handcalc
import forallpeople as si
from numba import njit
from steel_lib.experimental_functions import calculation_lc,bolt_tearout,bolt_bearing
from numpy import atan,radians
# def min()
@handcalc(jupyter_display=True)
def calculation_lc(theta,L,S,d_hole):
    if theta != 0:val1 = L/sin(theta) - 0.5*d_hole;val2 = S/cos(S) - d_hole;lc = min(val1,val2)
    elif theta == 0: lc = S/cos(S) - d_hole
    return lc
calculation_lc(13,12*si.m,12*si.m,12*si.m)
@njit
def test_min_max():
    return min(1,2)
test_min_max()
calculation_lc(12,123,123,123)
bolt_tearout(123,123,123,123,123,123)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

274663969.2

In [1]:
from math import sin,cos,pi
from handcalcs.decorator import handcalc
import forallpeople as si
from numba import njit
from steel_lib.experimental_functions import bolt_bearing,block_shear
from numpy import atan,radians

ao1_angle_bearing_args = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.375,
    "P_u": 5,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.25,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75,
    "c": 4
}
ao1_beam_bearing_args = {
    "F_u": 65,
    "d_bolt": 7/8,
    "t": 0.25,
    "P_u": 5,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.75,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75
}
ao1_0sl_angle_bearing_args = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.375,
    "P_u": 0,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.25,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75,
    'c':4
}

a02_osl_bearing = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.5,
    "P_u": 0,
    "V_u": 70,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.428,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75,
    'c':4
}

a03_nsl_plate_bearing = {
    "F_u": 65,
    "d_bolt": 1+1/8,
    "t": 0.625,
    "P_u": 60,
    "V_u": 90,
    "S_r": 3,
    "N_r": 5,
    "S_c": 3,
    "N_c": 3,
    "L_ev": 1.5,
    "L_eh": 2,
    "dv":  1.25,
    "dh": 1.5,
    "phi": 0.75,
    'c':5.226
}




    
block_shear(P_u = 60, V_u = 90, F_y = 50, F_u = 65, t = 0.625, N_r =5, S_r = 3, N_c = 3 , S_c = 3, L_ev = 1.5, L_eh = 2, d_v = 1.25,d_h=1.5, phi = 0.75)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

123

In [7]:
%%timeit
bolt_bearing(**ao1_0sl_angle_bearing_args)

2.07 μs ± 43.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
